## CellCycleNet Example - Predict cell cycle stage from 2D DAPI images WITHOUT ground truth labels.
This notebook demonstrates how to use CellCycleNet to predict cell cycle stage from images of DAPI-stained nuclei that do not have associated ground truth labels for cell cycle stage.

CellCycleNet requires the following data:
 - A directory of 2D DAPI-stained fields of view named as `tile_<tile_num>.tiff`
 - A directory of 2D segmentation masks named as `mask_<tile_num>.tiff`
 - Where `<tile_num>` is an integer that uniquely identifies each field of view and its corresponding segmentation mask

### Step 1: Create single-nucleus images from segmented FOVs.

In [ ]:
from cellcyclenet import utils

IMAGE_DIR = '../example_data/2d/test_tiles/' # path to DAPI-stained FOVs
MASK_DIR = '../example_data/2d/test_masks/' # path to segmentation masks of FOVs
OUTPUT_DIR = '../example_data/2d/test_SNI/' # path where single-nucleus images will be saved

# generate unlabeled SNIs #
'''
Optional Arguments for utils.generate_images():
    - return_df: boolean, if True, returns a dataframe with the paths to the generated SNIs; dataframe will saved as a .csv file in any case
    - num_cores: integer, number of cores to use for parallel processing; if 'None', no parallel processing will be used
    - is_3d: boolean, set to True if your data is 3D; set to False if your data is 2D
'''
df = utils.generate_images(IMAGE_DIR, MASK_DIR, OUTPUT_DIR, return_df=True, num_cores=None, is_3d=False)
df

### Step 2: Predict cell cycle stage for each single-nucleus image.

In [ ]:
from cellcyclenet import CellCycleNet

# create 3D model instance (pretrained weights are loaded by default) #
model = CellCycleNet(is_3d=False)

# convert dataframe to CellCycleNet dataset; split_data=False --> no validation or testing sets will be created #
dataset = model.create_dataset(dataframe=df, split_data=False)

# generate cell cycle stage predictions (0 = G1, 1 = S/G2); with_labels=False --> predicting on unlabeled data #
predictions = model.predict(dataset, with_labels=False)

# display predictions #
predictions.sort_values(['tile_num', 'obj_num'])
predictions